# Automated Crop Disease Detection and Classification Using Deep CNN
Team: Krishok_Er_Dristy | CSE 422 Neural Networks Lab

Run on Colab: **Runtime -> Change runtime type -> T4 GPU**. Run the cells from top to bottom.

- Part A: Data (Asif)
- Part B: Models (Jakaria)
- Part C: Training (Tamim)
- Part D: Evaluation, Grad-CAM, demo (Shafayat)

## Part A: Data pipeline (Asif)
### Step 1: Setup

In [ ]:
import os, json, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - change runtime to T4 GPU")

# Save everything to Google Drive so nothing is lost if Colab disconnects
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = Path("/content/drive/MyDrive/crop_disease")
CKPT_DIR = OUT_DIR / "checkpoints"
FIG_DIR = OUT_DIR / "figures"
for d in (OUT_DIR, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Saving to:", OUT_DIR)

### Step 2: Download and inspect the dataset

In [ ]:
%pip install -q kagglehub gradio
import kagglehub
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("Path:", path)

for p in sorted(Path(path).glob("*")):
    print(p.name)
    for q in sorted(p.glob("*"))[:5]:
        print("   ", q.name)

### Step 3: Build dataframe (Potato, Tomato, Corn only) and split 70/15/15

In [ ]:
CROPS = ("potato", "tomato", "corn")
EXTS = {".jpg", ".jpeg", ".png"}

rows = []
for f in Path(path).rglob("*"):
    if f.suffix.lower() in EXTS:
        cls = f.parent.name                      # class = folder name
        if cls.lower().startswith(CROPS):
            rows.append((str(f), cls))

df = pd.DataFrame(rows, columns=["path", "cls"])
classes = sorted(df.cls.unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df["label"] = df.cls.map(class_to_idx)
NUM_CLASSES = len(classes)
print(NUM_CLASSES, "classes,", len(df), "images")
print(df.cls.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df.label, random_state=SEED)
print("train/val/test:", len(train_df), len(val_df), len(test_df))

# Save the split so the whole team uses the identical split
train_df.to_csv(OUT_DIR / "train.csv", index=False)
val_df.to_csv(OUT_DIR / "val.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)
json.dump(classes, open(OUT_DIR / "classes.json", "w"))

# If the runtime restarts: re-run Steps 1-2 (to restore the dataset path), then run this instead of the cell above:
# train_df = pd.read_csv(OUT_DIR/"train.csv"); val_df = pd.read_csv(OUT_DIR/"val.csv"); test_df = pd.read_csv(OUT_DIR/"test.csv")
# classes = json.load(open(OUT_DIR/"classes.json")); NUM_CLASSES = len(classes)

### Step 4: Transforms, Dataset, DataLoaders

In [ ]:
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),      # slight zoom
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.2, contrast=0.2),     # lighting variation
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class LeafDataset(Dataset):
    def __init__(self, df, tf):
        self.paths = df.path.tolist()
        self.labels = df.label.astype(int).tolist()
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.labels[i]

BS = 32
train_dl = DataLoader(LeafDataset(train_df, train_tf), batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(LeafDataset(val_df,   eval_tf),  batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(LeafDataset(test_df,  eval_tf),  batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

xb, yb = next(iter(train_dl))
print(xb.shape, yb.shape)

### Step 5: EDA (for the report)

In [ ]:
df.cls.value_counts().sort_values().plot(kind="barh", figsize=(9, 7), title="Images per class")
plt.tight_layout(); plt.savefig(FIG_DIR / "class_distribution.png", dpi=150); plt.show()

mean_a, std_a = np.array(MEAN), np.array(STD)
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for a, img, lab in zip(ax.flat, xb[:8], yb[:8]):
    im = img.permute(1, 2, 0).numpy() * std_a + mean_a
    a.imshow(np.clip(im, 0, 1)); a.set_title(classes[lab].replace("___", "\n"), fontsize=8); a.axis("off")
plt.suptitle("Augmented training samples")
plt.tight_layout(); plt.savefig(FIG_DIR / "samples.png", dpi=150); plt.show()

## Part B: Models (Jakaria)
### Step 6: Baseline custom CNN (from scratch)

In [ ]:
class BaselineCNN(nn.Module):
    """4 conv blocks: Conv -> BatchNorm -> ReLU -> MaxPool, then a linear classifier."""
    def __init__(self, num_classes):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=False),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

print(sum(p.numel() for p in BaselineCNN(NUM_CLASSES).parameters()) / 1e6, "M parameters")

### Step 7: Transfer learning models (ResNet-50, EfficientNet-B0)
The final layer outputs raw scores (logits). The Softmax is applied inside `CrossEntropyLoss` during training
and with `.softmax()` at prediction time.

In [ ]:
from torchvision import models

def build_model(name, num_classes, pretrained=True, freeze_backbone=True):
    if name == "baseline_cnn":
        return BaselineCNN(num_classes)

    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        if freeze_backbone:
            for p in m.parameters(): p.requires_grad = False
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, num_classes))
        return m

    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        if freeze_backbone:
            for p in m.parameters(): p.requires_grad = False
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, num_classes))
        return m

    raise ValueError(name)

def unfreeze_last_block(model, name):
    """For fine-tuning: unfreeze only the last block(s) of the backbone."""
    if name == "resnet50":
        for p in model.layer4.parameters(): p.requires_grad = True
    elif name == "efficientnet_b0":
        for p in model.features[6:].parameters(): p.requires_grad = True
    return model

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6

for n in ["baseline_cnn", "resnet50", "efficientnet_b0"]:
    print(f"{n}: trainable params = {count_trainable(build_model(n, NUM_CLASSES)):.3f} M")

## Part C: Training (Tamim)
### Step 8: Reusable training function
CrossEntropyLoss + AdamW + ReduceLROnPlateau + EarlyStopping. The best weights (lowest validation loss) are saved to Drive.

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

HISTORY = {}   # name -> dict of lists

def run_epoch(model, dl, criterion, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=(device.type == "cuda")):
                out = model(x)
                loss = criterion(out, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

def fit(model, name, epochs=20, lr=1e-3, patience=5, weight_decay=1e-4):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    best_loss, bad_epochs = float("inf"), 0
    ckpt = CKPT_DIR / f"{name}.pth"

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, train_dl, criterion, optimizer, scaler)
        vl, va = run_epoch(model, val_dl, criterion)
        scheduler.step(vl)
        cur_lr = optimizer.param_groups[0]["lr"]
        for k, v in zip(hist.keys(), (tl, vl, ta, va, cur_lr)):
            hist[k].append(v)
        print(f"[{name}] epoch {ep:02d}/{epochs} | train loss {tl:.4f} acc {ta:.4f} | "
              f"val loss {vl:.4f} acc {va:.4f} | lr {cur_lr:.1e} | {time.time()-t0:.0f}s")

        if vl < best_loss - 1e-4:
            best_loss, bad_epochs = vl, 0
            torch.save(model.state_dict(), ckpt)          # save best to Drive
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping at epoch {ep}")
                break

    model.load_state_dict(torch.load(ckpt, map_location=device))   # restore best weights
    HISTORY[name] = hist
    json.dump(hist, open(CKPT_DIR / f"{name}_history.json", "w"))
    print(f"Best val loss {best_loss:.4f} -> saved {ckpt}")
    return model

### Step 9: Train all models
Order: baseline CNN -> ResNet-50 (head, then fine-tune) -> EfficientNet-B0 (head, then fine-tune).
If Colab disconnects, re-run Steps 1-8, then continue from the model that did not finish. Saved checkpoints stay on Drive.

In [ ]:
# 9a. Baseline CNN (trained from scratch)
baseline = build_model("baseline_cnn", NUM_CLASSES)
baseline = fit(baseline, "baseline_cnn", epochs=25, lr=1e-3, patience=5)

In [ ]:
# 9b. ResNet-50: phase 1 (train only the new head)
resnet = build_model("resnet50", NUM_CLASSES)
resnet = fit(resnet, "resnet50_head", epochs=8, lr=1e-3, patience=3)

In [ ]:
# 9c. ResNet-50: phase 2 (fine-tune layer4 with a small learning rate)
resnet = unfreeze_last_block(resnet, "resnet50")
print("Trainable params:", round(count_trainable(resnet), 2), "M")
resnet = fit(resnet, "resnet50_ft", epochs=10, lr=1e-4, patience=4)

In [ ]:
# 9d. EfficientNet-B0: phase 1 (head only)
effnet = build_model("efficientnet_b0", NUM_CLASSES)
effnet = fit(effnet, "effnet_head", epochs=8, lr=1e-3, patience=3)

In [ ]:
# 9e. EfficientNet-B0: phase 2 (fine-tune last blocks)
effnet = unfreeze_last_block(effnet, "efficientnet_b0")
print("Trainable params:", round(count_trainable(effnet), 2), "M")
effnet = fit(effnet, "effnet_ft", epochs=10, lr=1e-4, patience=4)

## Part D: Evaluation and explainability (Shafayat)
### Step 10: Load best models and evaluate on the test set

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, precision_recall_curve)
from sklearn.preprocessing import label_binarize
import seaborn as sns

# Final models = best saved checkpoints (works even after a runtime restart)
FINAL = {
    "Baseline CNN":    ("baseline_cnn",    "baseline_cnn"),
    "ResNet-50":       ("resnet50",        "resnet50_ft"),
    "EfficientNet-B0": ("efficientnet_b0", "effnet_ft"),
}
final_models = {}
for label, (arch, ckpt_name) in FINAL.items():
    m = build_model(arch, NUM_CLASSES, pretrained=False, freeze_backbone=False)
    m.load_state_dict(torch.load(CKPT_DIR / f"{ckpt_name}.pth", map_location=device))
    final_models[label] = m.to(device).eval()

@torch.no_grad()
def predict_all(model, dl):
    model.eval()
    ys, probs = [], []
    for x, y in dl:
        probs.append(model(x.to(device)).softmax(1).cpu())
        ys.append(y)
    probs = torch.cat(probs).numpy(); ys = torch.cat(ys).numpy()
    return ys, probs.argmax(1), probs

def latency_ms(model, n=100):
    """Average time (ms) for one single-image forward pass."""
    x = torch.randn(1, 3, 224, 224, device=device)
    with torch.no_grad():
        for _ in range(10): model(x)                       # warm-up
        if device.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(n): model(x)
        if device.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / n * 1000

preds = {}
results = []
for label, m in final_models.items():
    y, p, pr = predict_all(m, test_dl)
    preds[label] = (y, p, pr)
    prec, rec, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    results.append({
        "Model": label,
        "Accuracy": accuracy_score(y, p),
        "Precision (macro)": prec,
        "Recall (macro)": rec,
        "Macro F1": f1,
        "Latency (ms/img)": latency_ms(m),
        "Params (M)": sum(q.numel() for q in m.parameters()) / 1e6,
    })

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df.to_csv(OUT_DIR / "model_comparison.csv")
results_df

In [ ]:
# Per-class report for each model
for label, (y, p, _) in preds.items():
    print("=" * 70); print(label); print("=" * 70)
    print(classification_report(y, p, target_names=classes, digits=4, zero_division=0))

In [ ]:
# Confusion matrices
for label, (y, p, _) in preds.items():
    cm = confusion_matrix(y, p)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, cbar=False)
    plt.title(f"Confusion matrix: {label}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=7)
    plt.tight_layout(); plt.savefig(FIG_DIR / f"confusion_{label.replace(' ', '_')}.png", dpi=150); plt.show()

In [ ]:
# Training curves (head phase + fine-tune phase joined for the transfer models)
def load_hist(name):
    return json.load(open(CKPT_DIR / f"{name}_history.json"))

curves = {
    "Baseline CNN":    [load_hist("baseline_cnn")],
    "ResNet-50":       [load_hist("resnet50_head"), load_hist("resnet50_ft")],
    "EfficientNet-B0": [load_hist("effnet_head"), load_hist("effnet_ft")],
}
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for label, parts in curves.items():
    tl = sum([h["train_loss"] for h in parts], []); vl = sum([h["val_loss"] for h in parts], [])
    ta = sum([h["train_acc"] for h in parts], []);  va = sum([h["val_acc"] for h in parts], [])
    ax[0].plot(tl, "--", label=f"{label} train"); ax[0].plot(vl, label=f"{label} val")
    ax[1].plot(ta, "--", label=f"{label} train"); ax[1].plot(va, label=f"{label} val")
ax[0].set_title("Loss"); ax[1].set_title("Accuracy")
for a in ax: a.set_xlabel("Epoch"); a.legend(fontsize=7); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "training_curves.png", dpi=150); plt.show()

In [ ]:
# Precision-Recall curves (micro-average over all classes)
plt.figure(figsize=(7, 6))
for label, (y, p, pr) in preds.items():
    y_bin = label_binarize(y, classes=range(NUM_CLASSES))
    prec, rec, _ = precision_recall_curve(y_bin.ravel(), pr.ravel())
    plt.plot(rec, prec, label=label)
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall curves (micro-average)")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "pr_curves.png", dpi=150); plt.show()

### Step 11: Grad-CAM

In [ ]:
import cv2

class GradCAM:
    """Grad-CAM: weight the target layer's feature maps by their average gradient, ReLU, normalize."""
    def __init__(self, model, layer):
        self.model = model.eval()
        self.acts, self.grads = None, None
        self.handle = layer.register_forward_hook(self._forward_hook)
    def _forward_hook(self, module, inp, out):
        self.acts = out.detach()
        out.register_hook(lambda g: setattr(self, "grads", g.detach()))
    def remove(self):
        self.handle.remove()
    def __call__(self, x, target=None):
        self.model.zero_grad()
        with torch.enable_grad():
            x = x.clone().requires_grad_(True)
            out = self.model(x)
            if target is None:
                target = out.argmax(1).item()
            out[0, target].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1))[0]
        cam = cam - cam.min()
        cam = (cam / (cam.max() + 1e-8)).cpu().numpy()
        return cam, target, out.softmax(1)[0].detach().cpu().numpy()

def target_layer(label, model):
    if label == "Baseline CNN":    return model.features[-1][2]   # last ReLU before final pooling
    if label == "ResNet-50":       return model.layer4[-1]
    if label == "EfficientNet-B0": return model.features[-1]

def overlay(img_pil, cam, alpha=0.4):
    img = np.array(img_pil.convert("RGB").resize((224, 224)))
    cam = cv2.resize(cam, (224, 224))
    heat = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return np.uint8((1 - alpha) * img + alpha * heat)

# Pick the best model by Macro F1
best_label = results_df["Macro F1"].idxmax()
best_model = final_models[best_label]
print("Best model:", best_label)
cam_engine = GradCAM(best_model, target_layer(best_label, best_model))

def show_gradcam(paths, true_labels=None, title="", save_name=None):
    n = len(paths)
    fig, ax = plt.subplots(2, n, figsize=(3 * n, 6.4))
    ax = np.array(ax).reshape(2, n)
    for i, pth in enumerate(paths):
        img = Image.open(pth).convert("RGB")
        x = eval_tf(img).unsqueeze(0).to(device)
        cam, pred, probs = cam_engine(x)
        ax[0, i].imshow(img.resize((224, 224))); ax[0, i].axis("off")
        ttl = f"pred: {classes[pred].replace('___', ' / ')}\n{probs[pred]*100:.1f}%"
        if true_labels is not None:
            ttl = f"true: {classes[true_labels[i]].replace('___', ' / ')}\n" + ttl
        ax[0, i].set_title(ttl, fontsize=7)
        ax[1, i].imshow(overlay(img, cam)); ax[1, i].axis("off")
    plt.suptitle(title); plt.tight_layout()
    if save_name: plt.savefig(FIG_DIR / save_name, dpi=150)
    plt.show()

In [ ]:
# 11a. Grad-CAM on correctly classified test images (random)
y_t, p_t, _ = preds[best_label]
rng = np.random.default_rng(SEED)
correct_idx = rng.choice(np.where(y_t == p_t)[0], size=6, replace=False)
show_gradcam(test_df.path.iloc[correct_idx].tolist(), test_df.label.iloc[correct_idx].tolist(),
             title=f"Grad-CAM ({best_label}): correct predictions", save_name="gradcam_correct.png")

In [ ]:
# 11b. Grad-CAM on misclassified test images (if any)
wrong_idx = np.where(y_t != p_t)[0]
print("Misclassified test images:", len(wrong_idx))
if len(wrong_idx) > 0:
    pick = wrong_idx[:6]
    show_gradcam(test_df.path.iloc[pick].tolist(), test_df.label.iloc[pick].tolist(),
                 title=f"Grad-CAM ({best_label}): mistakes", save_name="gradcam_wrong.png")

In [ ]:
# 11c. Real-world test: put 10-20 of your own phone photos of leaves in this Drive folder
REAL_DIR = OUT_DIR / "real_photos"
REAL_DIR.mkdir(exist_ok=True)
real_paths = [str(p) for p in sorted(REAL_DIR.iterdir()) if p.suffix.lower() in EXTS][:8]
print("Real photos found:", len(real_paths), "in", REAL_DIR)
if real_paths:
    show_gradcam(real_paths, title=f"Grad-CAM ({best_label}): real phone photos", save_name="gradcam_real.png")
else:
    print("Add photos to that folder and re-run this cell.")

### Step 12: Final comparison table and Gradio demo (bonus)

In [ ]:
print(results_df.to_string())
results_df[["Accuracy", "Macro F1"]].plot(kind="bar", figsize=(7, 4), ylim=(0.5, 1.0), title="Model comparison (test set)")
plt.xticks(rotation=0); plt.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "model_comparison.png", dpi=150); plt.show()

In [ ]:
import gradio as gr

def predict_demo(img):
    if img is None:
        return None, None
    pil = Image.fromarray(img).convert("RGB")
    x = eval_tf(pil).unsqueeze(0).to(device)
    cam, pred, probs = cam_engine(x)
    top = np.argsort(probs)[::-1][:3]
    label_dict = {classes[i].replace("___", " / "): float(probs[i]) for i in top}
    return label_dict, overlay(pil, cam)

demo = gr.Interface(
    fn=predict_demo,
    inputs=gr.Image(type="numpy", label="Upload a leaf image"),
    outputs=[gr.Label(num_top_classes=3, label="Diagnosis"), gr.Image(label="Grad-CAM heatmap")],
    title="Crop Disease Detection (Potato / Tomato / Corn)",
    description="Upload a leaf photo. The CNN predicts the disease and Grad-CAM shows where it looked.",
)
demo.launch(share=True)   # prints a public link you can open during the defense